In [ ]:
import numpy as np

def fit_exp_concave_two_points(x1, dT1, x2, dT2):
    """
    Fit a, b in dT(x) = a*(1 - exp(-b*x)) using two anchor points:
      dT(x1) = dT1
      dT(x2) = dT2
    Requires 0 < x1 < x2 and 0 < dT1 < dT2.
    Returns (a, b).

    x's can be canopy-fraction increases (0-1) or canopy-area increases,
    as long as you're consistent.
    """
    if not (0 < x1 < x2):
        raise ValueError("Need 0 < x1 < x2")
    if not (0 < dT1 < dT2):
        raise ValueError("Need 0 < dT1 < dT2")

    # Solve for b via 1D search (robust)
    # From ratio: (1 - e^{-b x1})/(1 - e^{-b x2}) = dT1/dT2
    target = dT1 / dT2

    def ratio(b):
        return (1 - np.exp(-b*x1)) / (1 - np.exp(-b*x2))

    # bracket b
    lo, hi = 1e-8, 1e6
    for _ in range(200):
        mid = np.sqrt(lo*hi)
        if ratio(mid) > target:
            hi = mid
        else:
            lo = mid
    b = np.sqrt(lo*hi)

    # Solve a from either point
    a = dT2 / (1 - np.exp(-b*x2))
    return a, b

# Example (fraction scale):
# "a ~ 2.0°C max-ish cooling by +30pp canopy" and "0.6°C by +10pp"
a, b = fit_exp_concave_two_points(x1=0.10, dT1=0.6, x2=0.30, dT2=2.0)
print(a, b)

In [ ]:
import numpy as np
import pandas as pd

# ---------- Heat Index (NWS) ----------
def heat_index_f(T_f, RH):
    """
    NWS Heat Index (°F) from temperature (°F) and RH (%).
    Vectorized for numpy arrays.
    """
    T = np.asarray(T_f, dtype=float)
    R = np.asarray(RH, dtype=float)

    # Simple approximation used for cooler conditions
    HI_simple = 0.5 * (T + 61.0 + (T - 68.0)*1.2 + R*0.094)
    HI = HI_simple.copy()

    mask = T >= 80.0
    if np.any(mask):
        Tm = T[mask]
        Rm = R[mask]

        HI_roth = (
            -42.379
            + 2.04901523*Tm
            + 10.14333127*Rm
            - 0.22475541*Tm*Rm
            - 0.00683783*Tm*Tm
            - 0.05481717*Rm*Rm
            + 0.00122874*Tm*Tm*Rm
            + 0.00085282*Tm*Rm*Rm
            - 0.00000199*Tm*Tm*Rm*Rm
        )

        # NWS adjustments
        adj_low = ((13 - Rm) / 4) * np.sqrt((17 - np.abs(Tm - 95)) / 17)
        low_mask = (Rm < 13) & (Tm >= 80) & (Tm <= 112)
        HI_roth[low_mask] -= adj_low[low_mask]

        adj_high = ((Rm - 85) / 10) * ((87 - Tm) / 5)
        high_mask = (Rm > 85) & (Tm >= 80) & (Tm <= 87)
        HI_roth[high_mask] += adj_high[high_mask]

        HI[mask] = HI_roth

    return HI

# ---------- Cooling curve ----------
def deltaT_concave(x, a, b):
    """
    ΔT(x) = a*(1 - exp(-b*x))
    x can be canopy fraction increase (0..1) or canopy area, depending on your choice.
    Returns ΔT in the same temperature unit as a (usually °C).
    """
    x = np.asarray(x, dtype=float)
    return a * (1.0 - np.exp(-b * x))

def c_to_f(T_c):
    return T_c * 9.0/5.0 + 32.0

# ---------- Derivative of HI w.r.t T (numeric) ----------
def dHI_dT_numeric(T_f, RH, eps=0.05):
    """
    Finite-difference derivative ∂HI/∂T at baseline (°F per °F).
    """
    T_f = np.asarray(T_f, dtype=float)
    RH = np.asarray(RH, dtype=float)
    return (heat_index_f(T_f + eps, RH) - heat_index_f(T_f - eps, RH)) / (2*eps)

# ---------- Tier builder ----------
def build_tiers(df_units, K=5, x_mode="fraction", unit_area_col=None, method="A"):
    """
    df_units must have columns:
      - 'unit_id'
      - 'T_base_f' (°F)
      - 'RH_base' (%)
      - 'A_plantable' (area units for decision variables)
      - 'a_i' (°C max cooling)
      - 'b_i' (depends on x_mode: 1/fraction or 1/area)

    Parameters:
      K: number of tiers
      x_mode: "fraction" or "area"
        - "fraction": x = (cumulative canopy area added) / (unit area)
          requires unit_area_col (e.g., total area of unit i in same units as A_plantable)
        - "area": x = cumulative canopy area added directly
      method: "A" or "B"
        - "A": compute HI at each breakpoint (exact)
        - "B": approximate using baseline derivative (faster)

    Returns a long dataframe with one row per (i,k).
    """
    required = {"unit_id", "T_base_f", "RH_base", "A_plantable", "a_i", "b_i"}
    missing = required - set(df_units.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    if x_mode == "fraction" and (unit_area_col is None or unit_area_col not in df_units.columns):
        raise ValueError("For x_mode='fraction', you must provide unit_area_col in df_units.")

    out_rows = []

    # Precompute baseline HI
    HI_base = heat_index_f(df_units["T_base_f"].values, df_units["RH_base"].values)

    # For method B: baseline derivative
    if method.upper() == "B":
        dH = dHI_dT_numeric(df_units["T_base_f"].values, df_units["RH_base"].values)
    else:
        dH = None

    for idx, row in df_units.iterrows():
        i = row["unit_id"]
        T0_f = float(row["T_base_f"])
        RH0 = float(row["RH_base"])
        Aplant = float(row["A_plantable"])
        a = float(row["a_i"])  # °C
        b = float(row["b_i"])

        if Aplant <= 0:
            # no tiers possible
            continue

        # tier widths (equal chunks)
        dA = Aplant / K
        # cumulative added canopy area after tier k
        X_area = np.array([dA * k for k in range(0, K+1)], dtype=float)  # includes 0

        # Map canopy added to x for ΔT
        if x_mode == "area":
            X_for_curve = X_area  # x is area
        else:
            unit_area = float(row[unit_area_col])
            X_for_curve = X_area / max(unit_area, 1e-12)  # x is fraction

        # Breakpoint cooling in °C then convert to °F
        dT_c = deltaT_concave(X_for_curve, a, b)  # length K+1
        dT_f = c_to_f(dT_c) - 32.0  # convert Δ°c to Δ°f correctly: ΔF = ΔC*9/5
        # (equivalently: dT_f = dT_c * 9/5)

        dT_f = dT_c * 9.0/5.0

        # Breakpoint temperatures
        T_break_f = T0_f - dT_f  # length K+1

        if method.upper() == "A":
            HI_break = heat_index_f(T_break_f, np.full_like(T_break_f, RH0))
        else:
            # Method B: use baseline derivative at T0,RH0
            # HI(x) ≈ HI0 - (dHI/dT)*ΔT
            HI0 = float(heat_index_f(np.array([T0_f]), np.array([RH0]))[0])
            dH0 = float(dHI_dT_numeric(np.array([T0_f]), np.array([RH0]))[0])
            HI_break = HI0 - dH0 * (T0_f - T_break_f)

        # Convert breakpoints into tier marginal values
        for k in range(1, K+1):
            dA_k = X_area[k] - X_area[k-1]
            dHI_k = HI_break[k-1] - HI_break[k]  # reduction (positive)
            beta_k = dHI_k / dA_k if dA_k > 0 else 0.0

            out_rows.append({
                "unit_id": i,
                "k": k,
                "dA_ik": dA_k,
                "X_ik_area": X_area[k],
                "x_for_curve": X_for_curve[k],
                "T_ik_f": T_break_f[k],
                "HI_ik_f": HI_break[k],
                "dHI_ik": dHI_k,
                "beta_ik": beta_k,
                "method": method.upper(),
                "x_mode": x_mode
            })

    return pd.DataFrame(out_rows)
